In [1]:
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.

In [3]:
from datasets import load_dataset, Dataset, concatenate_datasets
from google.colab import drive

In [4]:
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/DATA 266/Final Project"

Mounted at /content/drive


In [27]:
GSM = load_dataset("json", data_files=f"{project_path}/data/gsm.json")['train']
HSM = load_dataset("json", data_files=f"{project_path}/data/hsm.json")['train']

GSM, HSM


(Dataset({
     features: ['question', 'answer'],
     num_rows: 100
 }),
 Dataset({
     features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id'],
     num_rows: 100
 }))

In [16]:
ds_paths = {f"baseline.{model.split('-')[0]}.{dataset}":
            f"{project_path}/out/baseline-{model}-{dataset}.json"
            for model in ["Llama-3.2-3B-Instruct", "gemma-3-4b-it"]
            for dataset in ["gsm", "hsm"]}

ds_paths

{'baseline.Llama.gsm': '/content/drive/MyDrive/DATA 266/Final Project/out/baseline-Llama-3.2-3B-Instruct-gsm.json',
 'baseline.Llama.hsm': '/content/drive/MyDrive/DATA 266/Final Project/out/baseline-Llama-3.2-3B-Instruct-hsm.json',
 'baseline.gemma.gsm': '/content/drive/MyDrive/DATA 266/Final Project/out/baseline-gemma-3-4b-it-gsm.json',
 'baseline.gemma.hsm': '/content/drive/MyDrive/DATA 266/Final Project/out/baseline-gemma-3-4b-it-hsm.json'}

In [18]:
ds_dict = load_dataset("json", data_files=ds_paths)
ds_dict

DatasetDict({
    baseline.Llama.gsm: Dataset({
        features: ['responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.Llama.hsm: Dataset({
        features: ['responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.gemma.gsm: Dataset({
        features: ['responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.gemma.hsm: Dataset({
        features: ['responses', 'time_deltas'],
        num_rows: 100
    })
})

In [30]:
name = 'baseline.gemma.hsm'
concatenate_datasets([HSM, ds_dict[name]], axis=1)

Dataset({
    features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'responses', 'time_deltas'],
    num_rows: 100
})

In [31]:
for name, ds in ds_dict.items():
    if "gsm" in name:
        ds_dict[name] = concatenate_datasets([GSM, ds], axis=1)
    elif "hsm" in name:
        ds_dict[name] = concatenate_datasets([HSM, ds], axis=1)
    else:
        ds_dict[name] = concatenate_datasets([Arithmetics, ds], axis=1)

ds_dict

DatasetDict({
    baseline.Llama.gsm: Dataset({
        features: ['question', 'answer', 'responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.Llama.hsm: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.gemma.gsm: Dataset({
        features: ['question', 'answer', 'responses', 'time_deltas'],
        num_rows: 100
    })
    baseline.gemma.hsm: Dataset({
        features: ['problem', 'solution', 'answer', 'subject', 'level', 'unique_id', 'responses', 'time_deltas'],
        num_rows: 100
    })
})